### Middleware
    Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
    - Tracking agent behavior with logging, analytics, and debugging.
    - Transforming prompts, tool selection, and output formatting.
    - Adding retries, fallbacks, and early termination logic.
    - Applying rate limits, guardrails, and PII detection.
#### Prebuilt middleware
    LangChain and Deep Agents provide prebuilt middleware for common use cases. Each middleware is production-ready and configurable for your specific needs.
         - Summarization
         - Human-in-the-loop
         etc.
#### Custom middleware
    Build custom middleware by implementing hooks that run at specific points in the agent execution flow.
         - Node-style hooks
         - Wrap-style hooks

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Message based Summarization Middleware
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("messages",10), # trigger summarization after 10 messages
            keep=("messages", 4) # keep the last 4 messages in memory after summarization
        )
        ]
)

In [3]:
### Run with Thread id
config = {"configurable": {"thread_id": "test-1"}}

In [4]:
# Alternative test data
questions = [
    "what is 2 + 2?",
    "what is 10 * 3?",
    "what is 30 / 5?",
    "what is 15 - 5?",
    "what is 6 * 6?",
    "what is 100 / 5?",
]

for quest in questions:
    response = agent.invoke({"messages": [HumanMessage(content=quest)]}, config=config)
    print(f"Messages: {response}")
    print(f"Messages:{len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 2 + 2?', additional_kwargs={}, response_metadata={}, id='ff2d5881-7598-4492-bae9-5a8ee788d4e0'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 15, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_88876bec1e', 'id': 'chatcmpl-Dygc9e47ldoobuSuwQxb48eGVHX2V', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f384a-e2fe-7af0-8659-c6356b3d5ade-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details'

### Token Size SummarizationMiddleware

In [5]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(location: str) -> str:
    """Search for hotels - returns a long response which consumes more tokens."""
    return f"""Hotels in {location}:
    1. Grand Hotel - 5 stars, $380 per night, pool, spa, free breakfast
    2. Hotel B - 4 stars, $250 per night, gym, free parking
    3. Hotel C - 3 stars, $150 per night, free Wi-Fi """

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 350),  # trigger summarization after 350 tokens
            keep=("tokens", 200)  # keep the last 200 tokens in memory after summarization
        ),
    ]
)

config = {"configurable": {"thread_id": "test-2"}}

## Token counter
def count_tokens(messages):
    total_characters = sum(len(str(message.content)) for message in messages)
    return total_characters

In [6]:
## Test Run
locations = ["New York", "Los Angeles", "Chicago", "Houston", "Miami","Dubai","Singapore"]

for city in locations:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Search hotels in {city}")]}, 
        config=config
    )
    
    Tokens = count_tokens(response["messages"])
    print(f"{city}: ~{Tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

New York: ~619 tokens, 4 messages
[HumanMessage(content='Search hotels in New York', additional_kwargs={}, response_metadata={}, id='947ac9b9-98c0-40af-b21c-6a0f9fc947a3'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 56, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0d8f5eee12', 'id': 'chatcmpl-Dyh96Y5Vf5qkltz0wGfBNfMkgBoU8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f386a-0e7b-7a91-8581-7e3f8b90af18-0', tool_calls=[{'name': 'search_hotels', 'args': {'location': 'New York'}, 'id': 'call_rJPLHibfcEoX6RYh5qUMkiTy', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'in

#### SummarizationMiddleware by Fraction
Utilize the given fraction of total token capacity of model

In [7]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(location: str) -> str:
    """Search for hotels - returns a long response which consumes more tokens."""
    return f"""Hotels in {location}:
    1. Grand Hotel - 5 stars, $380 per night, pool, spa, free breakfast
    2. Hotel B - 4 stars, $250 per night, gym, free parking
    3. Hotel C - 3 stars, $150 per night, free Wi-Fi """

agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # trigger summarization after 0.5% of the total tokens in memory
            keep=("fraction", 0.002)  # keep the last 0.2% of tokens in memory after summarization
        ),
    ]
)

config = {"configurable": {"thread_id": "test-3"}}

## Token counter
def count_tokens(messages):
    total_characters = sum(len(str(message.content)) for message in messages)
    return total_characters

In [ ]:
## Test Run
locations = ["Houston", "Miami","Dubai","Singapore"]

for city in locations:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Search hotels in {city}")]}, 
        config=config
    )
    
    Tokens = count_tokens(response["messages"])
    fraction = Tokens / 128000  # Assuming 128000 is the total number of tokens in memory 
    print(f"{city}: ~{Tokens} tokens ({fraction:0.2%}), {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Houston: ~623 tokens (62.30%), 4 messages
[HumanMessage(content='Search hotels in Houston', additional_kwargs={}, response_metadata={}, id='5a9b47f6-31ed-4db9-8a37-a9ef01945c5a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 55, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0d8f5eee12', 'id': 'chatcmpl-Dyhmg4KDYaEqpwZWEWj1CVdbILrwe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f388f-8234-7392-abff-ad06ca90a2fa-0', tool_calls=[{'name': 'search_hotels', 'args': {'location': 'Houston'}, 'id': 'call_x81AoUDWaoJii0oXX5wDTDjn', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat